In [ ]:
# src/pipeline.py
# Unified Knowledge-Gap Mapping Pipeline

import pandas as pd
from src.ingestion import LogIngestion
from src.features import FeatureEngineer
from src.model import KnowledgeGapModel
from src.evaluation import GapEvaluator

class KnowledgeGapPipeline:
    """Complete end-to-end pipeline for the Lightweight Knowledge-Gap Mapping System"""
    
    def __init__(self):
        self.ingestor = LogIngestion()
        self.engineer = FeatureEngineer()
        self.model = KnowledgeGapModel(max_depth=5)
        self.evaluator = GapEvaluator()
        self.is_trained = False
        self.feature_df = None
        self.predictions = None
        self.student_map = None
    
    def run_full_pipeline(self, use_sample=True):
        """Run the entire pipeline from raw data to knowledge gap map"""
        print("🚀 Starting Full Knowledge-Gap Mapping Pipeline...\n")
        
        # 1. Data Ingestion
        print("Step 1: Loading and Standardizing Data...")
        if use_sample:
            raw_df = self.ingestor.get_sample_data()
        else:
            raw_df = self.ingestor.load_oulad_vle()
        
        enriched = self.ingestor.enrich_with_activity_type(raw_df)
        std_df = self.ingestor.standardize_logs(enriched)
        
        print(f"   → Standardized data shape: {std_df.shape}\n")
        
        # 2. Feature Engineering
        print("Step 2: Feature Engineering...")
        student_info = None
        try:
            student_info = pd.read_csv("data/studentInfo.csv")
            student_info = student_info[
                (student_info['code_module'] == 'AAA') & 
                (student_info['code_presentation'] == '2013J')
            ]
        except:
            print("   Warning: Could not load studentInfo.csv for target labels")
        
        self.feature_df = self.engineer.build_full_feature_set(std_df, student_info)
        print(f"   → Feature DataFrame shape: {self.feature_df.shape}\n")
        
        # 3. Model Training
        print("Step 3: Training Model...")
        self.model.train(self.feature_df)
        self.is_trained = True
        print("   → Model training completed\n")
        
        # 4. Knowledge Gap Prediction
        print("Step 4: Inferring Knowledge Gaps...")
        self.predictions = self.model.predict_gaps(self.feature_df)
        
        # 5. Evaluation & Mapping
        print("Step 5: Generating Knowledge Gap Map + Remediation...")
        self.enhanced_predictions, self.student_map = self.evaluator.create_knowledge_gap_map(self.predictions)
        
        print("\n✅ FULL PIPELINE COMPLETED SUCCESSFULLY!")
        print(f"Processed {self.feature_df['student_id'].nunique()} students")
        print(f"Detected knowledge gaps across {len(self.enhanced_predictions)} student-concept pairs")
        
        return self.enhanced_predictions, self.student_map
    
    def get_student_gap_map(self, student_id):
        """Get detailed gap map for a specific student"""
        if self.enhanced_predictions is None:
            raise ValueError("Run the pipeline first!")
        return self.enhanced_predictions[self.enhanced_predictions['student_id'] == student_id]
    
    def save_results(self):
        """Save results for later use in dashboard"""
        import os
        os.makedirs("results", exist_ok=True)
        self.enhanced_predictions.to_csv("results/enhanced_predictions.csv", index=False)
        self.student_map.to_csv("results/student_gap_summary.csv", index=False)
        print("Results saved to /results/ folder")


# ========================== INTEGRATION TEST ==========================
if __name__ == "__main__":
    print("=== DAY 6: Testing Unified Pipeline ===\n")
    
    pipeline = KnowledgeGapPipeline()
    enhanced, summary = pipeline.run_full_pipeline(use_sample=False)
    
    print("\n=== TOP STUDENTS WITH HIGHEST KNOWLEDGE GAPS ===")
    print(summary.sort_values('gap_percentage', ascending=False).head(10))
    
    # Test single student view
    sample_student = summary['student_id'].iloc[0]
    print(f"\nDetailed view for student {sample_student}:")
    print(pipeline.get_student_gap_map(sample_student)[['concept', 'severity', 'evidence', 'remediation_suggestions']].head(6))
    
    pipeline.save_results()
    
    print("\n🎉 DAY 6 SUCCESS! The complete pipeline is now integrated and working end-to-end.")

In [ ]:
# 06_pipeline_integration_test.ipynb

from src.pipeline import KnowledgeGapPipeline

print("=== DAY 6: Full Pipeline Integration Test ===\n")

pipeline = KnowledgeGapPipeline()

# Run the complete pipeline
enhanced_predictions, student_summary = pipeline.run_full_pipeline(use_sample=False)

print("\n" + "="*60)
print("PIPELINE SUMMARY")
print("="*60)
print(f"Total Students Processed     : {student_summary.shape[0]}")
print(f"Students with High Risk      : {(student_summary['risk_level'] == 'High').sum()}")
print(f"Average Gap Percentage       : {student_summary['gap_percentage'].mean():.1f}%")

print("\nTop 5 Students by Gap Percentage:")
print(student_summary.sort_values('gap_percentage', ascending=False)[['student_id', 'gap_percentage', 'risk_level']].head(5))

print("\n🎉 DAY 6 COMPLETE! All modules are now working together smoothly.")